# Bake off redux: a review and experimental evaluation of recent time series classification algorithms

Usage example for the paper "Bake off redux: a review and experimental evaluation of recent time series classification algorithms" published in Data Mining and Knowledge Discovery.

See the [README](README.md) for the results files, datasets, installation, experiment scripts and classifier parameters.

## Using Classifiers

The classifiers used in our experiments extend the `scikit-learn` interface and can also be used like their estimators:

In [1]:
import warnings

warnings.filterwarnings("ignore")

from aeon.classification.interval_based import TimeSeriesForestClassifier
from sklearn.metrics import accuracy_score
from tsml.compose import SklearnToTsmlClassifier
from tsml.datasets import load_minimal_chinatown

from tsml_eval.utils.estimator_validation import is_sklearn_classifier

from set_bakeoff_classifier import _set_bakeoff_classifier

Data can be loaded using whichever method is most convenient, but should be formatted as either a 3D numpy array of shape (n_samples, n_channels, n_timesteps) or a list of length (n_samples) containing 2D numpy arrays of shape (n_channels, n_timesteps).

A function is available for loading from .ts files.

In [2]:
# load example classification dataset
X_train, y_train = load_minimal_chinatown("TRAIN")
X_test, y_test = load_minimal_chinatown("TEST")

# data can be loaded from .ts files using the following function
# from tsml.datasets import load_from_ts_file
# X, y = load_from_ts_file("data/data.ts")

print(type(X_train), type(y_train))
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)
X_train[:5]

<class 'numpy.ndarray'> <class 'numpy.ndarray'>
(20, 1, 24) (20,)
(20, 1, 24) (20,)


array([[[ 573.,  375.,  301.,  212.,   55.,   34.,   25.,   33.,  113.,
          143.,  303.,  615., 1226., 1281., 1221., 1081.,  866., 1096.,
         1039.,  975.,  746.,  581.,  409.,  182.]],

       [[ 394.,  264.,  140.,  144.,  104.,   28.,   28.,   25.,   70.,
          153.,  401.,  649., 1216., 1399., 1249., 1240., 1109., 1137.,
         1290., 1137.,  791.,  638.,  597.,  316.]],

       [[ 603.,  348.,  176.,  177.,   47.,   30.,   40.,   42.,  101.,
          180.,  401.,  777., 1344., 1573., 1408., 1243., 1141., 1178.,
         1256., 1114.,  814.,  635.,  304.,  168.]],

       [[ 428.,  309.,  199.,  117.,   82.,   43.,   24.,   64.,  152.,
          183.,  408.,  797., 1288., 1491., 1523., 1460., 1365., 1520.,
         1700., 1797., 1596., 1139.,  910.,  640.]],

       [[ 372.,  310.,  203.,  133.,   65.,   39.,   27.,   36.,  107.,
          139.,  329.,  651.,  990., 1027., 1041.,  971., 1104.,  844.,
         1023., 1019.,  862.,  643.,  591.,  452.]]])

Classifiers can be built using the `fit` method and predictions can be made using `predict`.

In [3]:
# build a TSF classifier and make predictions
tsf = TimeSeriesForestClassifier(n_estimators=100, random_state=0)
tsf.fit(X_train, y_train)
tsf.predict(X_test)

array(['1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '2', '2', '2',
       '2', '2', '1', '2', '2', '2', '2'], dtype='<U1')

`predict_proba` can be used to get class probabilities.

In [4]:
tsf.predict_proba(X_test)

array([[0.95, 0.05],
       [0.83, 0.17],
       [0.85, 0.15],
       [0.95, 0.05],
       [0.82, 0.18],
       [0.78, 0.22],
       [0.94, 0.06],
       [0.91, 0.09],
       [0.85, 0.15],
       [0.89, 0.11],
       [0.17, 0.83],
       [0.15, 0.85],
       [0.47, 0.53],
       [0.24, 0.76],
       [0.06, 0.94],
       [0.96, 0.04],
       [0.08, 0.92],
       [0.  , 1.  ],
       [0.  , 1.  ],
       [0.31, 0.69]])

Here we run some of the classifiers from the publication and find the accuracy for them on our example dataset.

In [5]:
classifiers = [
    "RDST",
    "DrCIF",
    "WEASEL-D",
    "MultiROCKET",
]

accuracies = []
for classifier_name in classifiers:
    # Select a classifier by name, see set_bakeoff_classifier.py for options
    classifier = _set_bakeoff_classifier(classifier_name, random_state=0)

    # if it is a sklearn classifier, wrap it to work with time series data
    if is_sklearn_classifier(classifier):
        classifier = SklearnToTsmlClassifier(
            classifier=classifier, concatenate_channels=True, random_state=0
        )

    # fit and predict
    classifier.fit(X_train, y_train)
    y_pred = classifier.predict(X_test)
    accuracies.append(accuracy_score(y_test, y_pred))

accuracies

[0.9, 0.95, 0.9, 0.85]